<i>École Polytechnique de Montréal</i>
<br>
<i>Cours PHS3104 : Mécanique quantique II</i>
<br>
<i>Session d'automne 2026</i>

<h3><center>Devoir 3 — Algorithme de Grover et coloriage de graphe</center></h3>

| Gabriel Poulin| Aiya Hamrouni | Nom Prénom |
|-------------------|-------------------|-------------------|
| 2379745 | 2364917| Matricule 0000000 |

In [7]:
%pip install qutip


     --------------------------------------- 10.8/10.8 MB 10.2 MB/s eta 0:00:00


**Initialisation du notebook**

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import Qobj, basis, tensor, qeye, expect
from qutip.measurement import measurement_statistics_povm

#### 4.1 Calculs analytiques [8 points]

**a) Phase et interférence [2 points]** — Pour le circuit de la figure 2a, avec $P(\phi) = \mathrm{diag}\left(1, e^{i\phi}\right)$, calculer l'état final et les probabilités de mesure après $P(\phi)H|0\rangle$, puis après $HP(\phi)H|0\rangle$. Montrer que la probabilité finale du résultat 1 vaut $\sin^2(\phi/2)$ et expliquer le rôle de la dernière porte de Hadamard.

**b) Action de la diffusion [2 points]** — Pour $|\psi\rangle = \sum_x a_x |x\rangle$, montrer que $D$ transforme chaque amplitude $a_x$ en $2\mu - a_x$, où $\mu = N^{-1}\sum_x a_x$.

**c) Une itération à la main [4 points]**  
**1.** Pour le triangle, identifier les chaînes valides et calculer les huit amplitudes après l'oracle, à partir de la superposition uniforme.

**2.** Calculer les amplitudes après une diffusion, donner le ket final et la probabilité totale de succès. Expliquer pourquoi les probabilités ne changent pas à l'étape de l'oracle.

#### 4.2 Construction des opérateurs avec QuTiP [15 points]

Vous devez créer les fonctions ci-dessous qui retourneront des `Qobj` sur les $n$ qubits, de taille $2^n \times 2^n$, avec `dims=[[2]*n, [2]*n]`.

**a) Projecteurs locaux [2 points]**  
**1.** Écrire `projecteur_local(n, k, b)`, qui retourne le projecteur $Q_k(b)$ défini en 3.2.

In [10]:
def projecteur_local(n,k,b):
    if k < 0 or k >= n:
        raise ValueError("Indice k invalide.")
    if b != 0 and b != 1: #b doit être 0 ou 1
        raise ValueError("b doit être égal à 0 ou 1.")
    facteurs = []
    for i in range(n):
        if i == k:
            facteurs.append(basis(2, b).proj())   
        else:
            facteurs.append(qeye(2))              
    return tensor(facteurs)

**2.** Pour $n = 3$, illustrer son action sur un état de base conservé et un état annulé.

In [ ]:
n = 3
Q = projecteur_local(n, 1, 0) 

conserve= basis([2]*n, [0, 0, 1])     
annule= basis([2]*n, [0, 1, 1])  

print(Q.dims, Q.shape)
print((Q * conserve - conserve).norm())
print((Q * annule).norm())

#L'application du projecteur sur l'état conservé
# donne Q_1(0)|001⟩ = |001⟩. L'état est conservé 
# et la norme de la différence est de 0.0.
#Pour l'état annulé la norme de la projection est nulle. Donc
# la projection annule l'état si le bit k ne vaut pas b.

[[2, 2, 2], [2, 2, 2]] (8, 8)
0.0
0.0


**b) Contraintes et oracle [7 points]**

**1.** Écrire `conflit(n, u, v, couleurs)`, qui retourne $C_{uv}$ à partir des listes de couleurs. [2 points]

In [14]:
def conflit(n,u,v,couleurs):
    I = tensor([qeye(2) for _ in range(n)])
    C_uv = 0 * I
    for i in (0, 1):
        for j in (0, 1):

            if couleurs[u][i] == couleurs[v][j]:

                Q_u = projecteur_local(n, u, i)
                Q_v = projecteur_local(n, v, j)

                C_uv += Q_u * Q_v

    return C_uv


**2.** Donner le projecteur obtenu pour AB de la maison comme exemple de vérification. [1 point]

In [16]:
                                                            # attendu : 8.0

**3.** Écrire `oracle(graphe)`, qui assemble $\Pi_{\text{val}}$ à partir des arêtes et retourne $O_f = I_N - 2\Pi_{\text{val}}$ sans programmer la liste des solutions (il devrait fonctionner pour la maison et le triangle). [2 points]

**4.** Justifier le fait que $O_f$ soit unitaire. Vérifier numériquement cette unitarité ainsi que le signe reçu par chaque état de base du triangle en utilisant une vérification indépendante des couleurs. [2 points]

**c) Diffusion et circuit complet [6 points]**

**1.** Écrire `diffusion(n)` à partir de $H^{\otimes n}$ et du projecteur sur $|0^n\rangle$. [2 points]

**2.** Écrire `circuit(graphe, r)`, qui retourne $U_r = \left(DO_f\right)^r H^{\otimes n}$, sans mesure finale. [2 points]

**3.** Pour le triangle, comparer l'état après une itération avec le résultat analytique. [2 points]

#### 4.3 Étude numérique de l'amplification [17 points]

**a) Validation classique [2 points]** — Trouver les chaînes valides et les couleurs correspondantes à chaque sommet pour le graphe maison. Cette liste servira à valider l'oracle et les résultats.

**b) Interférences et amplification [15 points]**  
**1.** Pour les deux graphes et $r = 0, 1, 2, 3, 4$, calculer $|\psi_r\rangle$ et tracer la probabilité totale de succès (la somme des probabilités pour les chaînes valides) en fonction de $r$. Comparer à la formule de l'encadré et identifier le point d'arrêt près du premier maximum. [6 points]

**2.** Pour la maison, tracer l'histogramme des 32 probabilités au point d'arrêt choisi et identifier les chaînes valides sur ce graphique. [3 points]

**3.** Expliquer brièvement comment l'oracle puis la diffusion produisent l'amplification, pourquoi les probabilités ne changent pas immédiatement après l'oracle et pourquoi un plus grand nombre d'itérations n'est pas toujours préférable. [6 points]